# Task 2 : Community detection

In [4]:
import networkx as nx
import pandas as pd
from networkx import community
import cdlib 

In [5]:
def build_graph_from_data(edges_path, posts_path):
    """
    Loads the datasets and builds a Graph where nodes are authors 
    and edges represent interactions (comments on posts).
    """
    # Load datasets
    df_edges = pd.read_csv(edges_path)
    df_posts = pd.read_csv(posts_path)
    
    # Standardizing columns: Linking commenters (Source) to post authors (Target)
    # Based on your previous logic, we ensure the graph captures user-to-user interaction
    G = nx.from_pandas_edgelist(
        df_edges, 
        source='Source', 
        target='Target', 
        create_using=nx.Graph()
    )
    
    print(f"Graph initialized: {G.number_of_nodes()} users, {G.number_of_edges()} interactions.")
    return G, df_posts

In [6]:
def detect_communities(G):
    """
    Implements the Louvain algorithm to identify modular communities.
    Returns a dictionary mapping {User: Community_ID}.
    """
    # Louvain algorithm for modularity optimization
    communities_list = nx.community.louvain_communities(G, seed=42)
    
    # Convert list of sets to a flat dictionary mapping
    community_mapping = {node: i for i, comm in enumerate(communities_list) for node in comm}
    
    print(f"Community Detection: Found {len(communities_list)} distinct groups.")
    return community_mapping

In [7]:
def calculate_centrality(G):
    """
    Calculates Betweenness (bridges) and PageRank (influencers).
    """
    print("Calculating Betweenness Centrality (using sampling for speed)...")
    # k=500 provides a good approximation for large graphs
    betweenness = nx.betweenness_centrality(G, k=500, seed=42)
    
    print("Calculating PageRank Centrality...")
    pagerank = nx.pagerank(G)
    
    return betweenness, pagerank

In [8]:
def identify_core_periphery(G):
    """
    Applies k-core decomposition to separate the highly engaged 
    'backbone' from casual participants.
    """
    # core_number returns the highest k-core each node belongs to
    core_numbers = nx.core_number(G)
    return core_numbers

In [9]:
def generate_analysis_report(G, df_posts):
    """
    Executes all analyses and merges them into a single analytical DataFrame.
    """
    # 1. Run Algorithms
    communities = detect_communities(G)
    betweenness, pagerank = calculate_centrality(G)
    core_scores = identify_core_periphery(G)
    
    # 2. Build Report
    report = pd.DataFrame({
        'User': list(G.nodes()),
        'Community_ID': [communities.get(node) for node in G.nodes()],
        'Betweenness_Score': [betweenness.get(node) for node in G.nodes()],
        'PageRank_Score': [pagerank.get(node) for node in G.nodes()],
        'Coreness_Level': [core_scores.get(node) for node in G.nodes()]
    })
    
    # 3. Label Core vs Periphery (Top 10% of coreness is considered the backbone)
    core_threshold = report['Coreness_Level'].quantile(0.9)
    report['Network_Role'] = report['Coreness_Level'].apply(
        lambda x: 'Core (Backbone)' if x >= core_threshold else 'Periphery'
    )
    
    return report

In [10]:
edges_file = 'graphs/all_edges_consolidated.csv'
posts_file = 'C:/Users/camil/Documents/IMT/3A/DaSci_UE/Graph_theory/data/all_posts_active_subreddit.csv'

G, df_posts = build_graph_from_data(edges_file, posts_file)
final_analysis = generate_analysis_report(G, df_posts)

print("\n--- Top 5 Key Influencers (PageRank) ---")
print(final_analysis.sort_values(by='PageRank_Score', ascending=False).head())

Graph initialized: 49994 users, 151771 interactions.
Community Detection: Found 270 distinct groups.
Calculating Betweenness Centrality (using sampling for speed)...
Calculating PageRank Centrality...

--- Top 5 Key Influencers (PageRank) ---
                   User  Community_ID  Betweenness_Score  PageRank_Score  \
23      ClimateShitpost            55           0.057705        0.006135   
420          idspispopd            19           0.044691        0.005412   
246   livinginahologram             0           0.055989        0.003818   
63   logicalprogressive           146           0.042982        0.003775   
229       suspended_007           146           0.025897        0.003739   

     Coreness_Level     Network_Role  
23               15  Core (Backbone)  
420              20  Core (Backbone)  
246              19  Core (Backbone)  
63               27  Core (Backbone)  
229              27  Core (Backbone)  
